# Z3 / SMT Solving

A self-contained refresher on **SMT (Satisfiability Modulo Theories)** solving with **Z3** — Microsoft Research's industrial-strength solver that decides logical formulas over rich theories (integers, reals, bit-vectors, arrays, strings) and either hands you a satisfying model or proves none exists.

**Domain:** Symbolic AI & Logic  ·  **recommended addition**  ·  **runnable:** yes (`z3-solver`, pure-Python bindings, CPU-only)

## 1. What & Why

**SAT** asks: is there an assignment of `true`/`false` to boolean variables that makes a formula true? **SMT** lifts that question from raw booleans to *theories* — instead of bare propositions you write constraints like `x + 2*y <= 10`, `a[i] == a[j]`, `(p & 0x0F) == 0x0A`, or `Length(s) > 3` — and the solver reasons about integers, reals, bit-vectors, arrays, and strings natively. **Z3** is the de-facto SMT solver: fast, free (MIT), and scriptable from Python.

You hand Z3 a set of constraints and ask `check()`. It answers one of three things:
- **`sat`** — the constraints are jointly satisfiable; call `model()` to get concrete values that work.
- **`unsat`** — no assignment can satisfy them all (this is how you *prove* things: assert the negation of a claim and get `unsat`).
- **`unknown`** — it gave up (hit a fragment that's undecidable or too hard, e.g. nonlinear integer arithmetic).

**The problem it solves.** A huge range of tasks reduce to "find values satisfying these constraints, or prove you can't": program verification (does this function ever divide by zero?), symbolic execution and fuzzing (what input reaches this branch?), test generation, type inference, scheduling and configuration, compiler optimization, and security/crypto analysis. Z3 is the engine inside many of them.

**Reach for it when** your problem is a conjunction of logical/arithmetic constraints over a decidable theory and you want either a witness or a proof of impossibility — especially verification, symbolic analysis, puzzles, and constraint problems with non-trivial logical structure.

**Skip it when** the work is pure numerical optimization over continuous variables (use an LP/MILP/convex solver like [[minizinc]]'s backends, OR-Tools, or scipy), is statistical/ML-shaped, or needs recursive *queries* over relational data ([[datalog]]) rather than constraint solving.

## 2. Mental Model

> **Z3 = a SAT solver that has been taught arithmetic, arrays, and bit-vectors. It searches boolean structure with CDCL, and every time it tentatively commits to a set of theory atoms it asks a specialist "theory solver" whether that combination is actually consistent.**

This architecture is called **CDCL(T)** — Conflict-Driven Clause Learning *modulo theories*:

```
        formula with theory atoms:  (x>0) ∧ (x<5 ∨ y=2) ∧ ...
                    │
                    ▼
   ┌─────────────────────────────────┐   propose a boolean assignment
   │   SAT core (CDCL): treats each   │   of the theory atoms
   │   atom like x>0 as a boolean     │ ─────────────────────────────►
   └─────────────────────────────────┘                     │
                    ▲                                       ▼
                    │                       ┌───────────────────────────────┐
     "that combo is impossible because…"    │  Theory solver(s): arithmetic, │
     learn a blocking clause, backtrack ◄─── │  bit-vectors, arrays, strings  │
                                            │  check real consistency        │
                                            └───────────────────────────────┘
```

The SAT engine handles the **boolean skeleton** (the and/or/not structure); the **theory solvers** handle the meaning of the atoms (`x + y <= 10` really is unsatisfiable with `x > 8 ∧ y > 8`). They ping-pong: the SAT core guesses, the theory solver refutes or confirms, conflicts get learned as new clauses, and the search narrows until a full consistent model emerges or the whole thing is proven `unsat`. You never see this loop — you just write constraints and read the verdict — but knowing it explains *why* nonlinear integer constraints can return `unknown` (that theory is undecidable) while linear ones are reliably decided.

## 3. Key Concepts

| Term | What it means |
|------|---------------|
| **SAT** | Boolean satisfiability — find a true/false assignment, or prove none exists. NP-complete; the engine underneath SMT. |
| **SMT** | SAT *Modulo Theories* — satisfiability of formulas whose atoms come from theories (arithmetic, arrays, bit-vectors, …). |
| **Theory** | A decision procedure for a class of constraints: `LIA`/`LRA` (linear int/real arithmetic), `NRA` (nonlinear real), `BV` (bit-vectors), `Array`, `String`, `UF` (uninterpreted functions). |
| **Sort** | A type in Z3: `Int`, `Real`, `Bool`, `BitVec(n)`, `Array(I, V)`, `String`. |
| **`sat` / `unsat` / `unknown`** | The three outcomes of `check()`: satisfiable (model available), unsatisfiable (proof of impossibility), or gave up. |
| **Model** | A concrete assignment witnessing `sat`. Retrieved with `s.model()`; evaluate terms with `m.eval(expr)`. |
| **Validity via unsat** | To prove `φ` holds for *all* inputs, assert `¬φ` and check: `unsat` means no counterexample exists, so `φ` is valid. |
| **`assert` / `add`** | Push a constraint into the solver's conjunction. The solver looks for a model of *all* asserted constraints together. |
| **Assumptions / `unsat core`** | Check under temporary assumptions; on `unsat`, ask for the minimal subset of assumptions that conflict — great for debugging over-constrained systems. |
| **`push` / `pop`** | Save/restore the assertion stack — explore a branch of constraints, then roll back (incremental solving). |
| **Optimize** | `z3.Optimize()` adds `maximize`/`minimize` objectives on top of satisfiability (MaxSMT / OMT). |
| **Tactics** | Composable preprocessing/solving strategies; the default `solver()` picks reasonable ones, but you can hand-tune for hard fragments. |

## 4. Setup

Z3 ships as a pip wheel that bundles the native solver — no separate binary, no build step. The Python package is `z3-solver`; you import it as `z3`.

```bash
pip install z3-solver
```

The cell below installs it (quietly, a no-op if already present) and prints the version. Everything in this notebook is CPU-only and runs in well under a second — no GPU, no network, no API key.

In [ ]:
%pip install -q z3-solver
import z3
print("z3 version:", z3.get_version_string())

## 5. Worked Examples

### Example 1 — Constraint satisfaction: find a model

The bread-and-butter use: state some constraints over integers and ask Z3 for values that satisfy them. Here we want two positive integers that sum to 12 where the second is the square of the first. `check()` returns `sat`, and `model()` hands back the witness. Note Z3 solves the algebra for you — there's no search loop to write.

In [ ]:
from z3 import Ints, Solver, sat

x, y = Ints('x y')
s = Solver()
s.add(x > 0, y > 0)        # domain
s.add(x + y == 12)         # they sum to 12
s.add(y == x * x)          # y is x squared

result = s.check()
print("check() ->", result)
if result == sat:
    m = s.model()
    print("model   ->", m)
    print(f"x={m[x]}, y={m[y]},  x+y={m[x].as_long()+m[y].as_long()}")

### Example 2 — Proving a theorem (and finding a counterexample)

This is the verification idiom that makes SMT powerful: **to prove a statement holds for *every* input, assert its negation and hope for `unsat`.** If Z3 can't find a single counterexample, the statement is valid.

First we prove the algebraic identity `(a + b)² = a² + 2ab + b²` over the reals: assert that the two sides *differ* and check — `unsat` means no `a, b` make them differ, i.e. they're always equal. Then we test a *false* claim (`a² < a` for all reals) and Z3 returns `sat` with a concrete counterexample, which is exactly what you want from a verifier that found a bug.

In [ ]:
from z3 import Reals, Solver, sat, unsat

a, b = Reals('a b')

# Claim 1 (true): (a+b)^2 == a^2 + 2ab + b^2 for all reals.
# Assert the NEGATION; unsat => no counterexample => the identity holds.
s = Solver()
s.add((a + b) * (a + b) != a*a + 2*a*b + b*b)
print("identity negated  ->", s.check(), "(unsat == proven for all a, b)")

# Claim 2 (false): a^2 < a for all reals. Assert it directly; sat => counterexample.
s2 = Solver()
s2.add(a*a < a)
print("a^2 < a           ->", s2.check())
if s2.check() == sat:
    print("counterexample    ->", s2.model(), " (e.g. fails for this a)")

### Example 3 — Optimization (OMT): the best model, not just any model

`z3.Optimize` is a superset of `Solver` that also takes objectives. This is a tiny integer linear program: maximize `3p + 2q` subject to a few linear constraints. Z3 returns the optimal model *and* the objective value — the same engine that does verification doubles as a (Max)SMT optimizer, handy when your constraints are logical/disjunctive in ways a pure LP solver can't express.

In [ ]:
from z3 import Optimize, Ints

p, q = Ints('p q')
opt = Optimize()
opt.add(p >= 0, q >= 0)     # non-negative
opt.add(p + q <= 10)        # budget
opt.add(p <= 2 * q)         # a ratio constraint

objective = opt.maximize(3 * p + 2 * q)
print("check() ->", opt.check())
print("optimal ->", opt.model())
print("max 3p+2q =", opt.upper(objective))

## 6. Gotchas & Pitfalls

- **`unknown` is a real answer.** Z3 isn't omniscient. Nonlinear *integer* arithmetic is undecidable, so constraints like `x*y == 6` over `Int` can return `unknown` (or hang). Linear arithmetic, bit-vectors, and nonlinear *reals* are decidable; integers + multiplication are the danger zone. Bound your variables, or move to bit-vectors when you can.
- **Python `and`/`or`/`not` silently do the wrong thing.** `x > 0 and x < 5` evaluates `x > 0` for truthiness in Python and returns the second expr — it does *not* build a conjunction. Use `z3.And(x > 0, x < 5)`, `z3.Or(...)`, `z3.Not(...)`, or pass multiple args to `s.add(...)`.
- **`==` on Z3 terms builds a constraint, it doesn't compare.** `x == y` is a symbolic equality expression, not a `bool`. Don't put Z3 terms in a plain `if`; use `is_true(m.eval(expr))` to read a model value as a Python bool.
- **Int vs Real vs BitVec changes everything.** `x / 2` is integer division on `Int`, exact on `Real`, and a fixed-width (wrapping!) operation on `BitVec`. Bit-vectors silently overflow (modular arithmetic) — that's a feature for modeling machine words, a trap if you meant unbounded integers.
- **Proving validity = asserting the negation.** A very common mistake is to assert the claim itself and conclude "sat, so it's proven." `sat` only means the claim is *satisfiable* (true for some input). To prove it for *all* inputs you must get `unsat` on its negation.
- **Unconstrained variables get arbitrary values.** If a variable appears in no constraint, the model picks anything (often `0`). A surprising model usually means a missing constraint, not a solver bug.
- **Quantifiers (`ForAll`/`Exists`) move you to a harder, often incomplete fragment.** Z3 handles them via heuristic instantiation (E-matching); results can be `unknown` and performance is fragile. Prefer quantifier-free formulations when possible.
- **Reuse and reset deliberately.** `s.add` accumulates forever. Use `push()`/`pop()` to scope temporary constraints, or build a fresh `Solver()` per query — otherwise stale assertions poison later checks.

## 7. When to Use vs Alternatives

| Option | Sweet spot | Trade-off vs Z3 |
|--------|-----------|-----------------|
| **Z3 / SMT** | Logical + arithmetic constraints over decidable theories; verification, symbolic execution, puzzles, where you want a model *or* a proof of impossibility | Extremely general and gives proofs; but can return `unknown` on undecidable fragments, and isn't a numerical optimizer at heart. |
| **Pure SAT (MiniSat, Glucose, CaDiCaL)** | Problems already in boolean CNF (planning, circuits) at massive scale | Faster on pure-boolean problems, but you must encode everything to bits yourself — Z3 gives you arithmetic/arrays for free. |
| **MILP / LP solvers (Gurobi, CBC, OR-Tools, scipy)** | Linear/continuous *optimization* with thousands of numeric variables | Far better at large numeric optimization; weaker at rich logical structure, disjunctions, and exact integer reasoning that SMT handles natively. |
| **[[minizinc]] / CP solvers** | Combinatorial constraint problems (scheduling, rostering, packing) with global constraints | Higher-level modeling and specialized propagators; SMT wins when you need theory reasoning (bit-vectors, arrays) or proofs of unsatisfiability. |
| **[[answer-set-programming]] (clingo)** | Finding *all* stable models, recursion through negation, combinatorial search | Different paradigm (declarative logic programs); ASP enumerates models, SMT decides one and reasons over rich theories. |
| **[[datalog]]** | Recursive *queries* over relational data | A deductive query engine, not a constraint solver — orthogonal job. |
| **Theorem provers (Coq, Lean, Isabelle)** | Deep mathematical proofs needing induction and higher-order logic | Far more expressive and fully trustworthy, but interactive and slow; SMT is the automated workhorse such provers *call* for decidable goals. |

**Rule of thumb:** if your problem is "satisfy these logical + arithmetic constraints, or prove you can't," Z3 is the default. If it's "optimize this big linear numeric model," use an LP/MILP solver. If it's "search combinatorially with global constraints," use CP/MiniZinc. SMT solvers (Z3 especially) are also the *engine* embedded inside symbolic-execution tools (angr, KLEE), verifiers (Dafny, Boogie), and many of the above.

## 8. Resources

- **Z3 Python API docs** (the `z3` module reference you'll keep open): <https://z3prover.github.io/api/html/namespacez3py.html>
- **Programming Z3** — the official, example-driven tutorial (Bjørner, de Moura, Nachmanson, Wintersteiger): <https://z3prover.github.io/papers/programmingz3.html>
- **Z3 GitHub** (source, releases, issues, the canonical wheel): <https://github.com/Z3Prover/z3>
- **The Z3 Guide** (interactive online tutorial, runs in the browser): <https://microsoft.github.io/z3guide/>
- **"Z3: An Efficient SMT Solver"** — the original TACAS 2008 paper: <https://link.springer.com/chapter/10.1007/978-3-540-78800-3_24>
- **SMT-LIB** — the standard input language and benchmark library for SMT solvers: <https://smt-lib.org/>

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def solve(clauses):
    """A satisfying assignment for the clause set, or None if there is none."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE